# Qwen3.8-27B — Colab T4 Quantization Benchmark

Bu notebook, **Qwen/Qwen3.8-27B** modelini Google Colab'daki **NVIDIA T4 (16 GB VRAM)** üzerinde
`bitsandbytes` ile **4-bit NF4** olarak yüklemeyi ve temel inference benchmarklarını ölçmeyi amaçlar.

> **Önemli:** 28B parametreli bir model 4-bit'te bile T4 için sınırdadır. Model ağırlıkları dışında
> quantization metadata'sı, quantize edilmeyen katmanlar, CUDA workspace ve KV cache de VRAM tüketir.
> Bu yüzden `CUDA out of memory` almak deneyin başarısız olduğu anlamına gelmez; projenin ölçmek istediği
> sınırın kendisidir.

Bu notebook modeli **Colab sunucusuna** indirir. Modeli kendi bilgisayarına indirmediğin sürece,
yaklaşık 15–60 GB'lık model trafiği kendi internet paketinden gitmez.

## 0. Colab ayarı

Colab menüsünden:

**Runtime → Change runtime type → T4 GPU**

seçili olmalı.

İlk çalıştırmada model dosyalarının Colab VM'ye indirilmesi uzun sürebilir.

In [ ]:
!nvidia-smi

## 1. Kütüphaneler

Qwen3.8 modeli Hugging Face üzerinde `AutoModelForMultimodalLM` ile örnekleniyor.
4-bit quantization için `bitsandbytes` kullanıyoruz.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

In [ ]:
import gc
import json
import math
import os
import time

import torch
import transformers
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU görünmüyor. Runtime → Change runtime type → T4 GPU seç.")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")

if "T4" not in props.name:
    print("UYARI: Bu notebook T4 hedeflenerek hazırlanmıştır; farklı GPU'da sonuçlar değişir.")

## 2. Teorik bellek referansı

Aşağıdaki hesap yalnızca **ağırlıkların ideal alt sınırını** gösterir.
Gerçek kullanım bundan daha yüksek olur.

In [ ]:
PARAMS_B = 28.0

for bits in [16, 8, 4, 3.5, 3]:
    gb_decimal = PARAMS_B * 1e9 * bits / 8 / 1e9
    gib = PARAMS_B * 1e9 * bits / 8 / 1024**3
    print(f"{bits:>4}-bit: yaklaşık {gb_decimal:5.2f} GB / {gib:5.2f} GiB sadece ağırlık")

## 3. 4-bit NF4 yapılandırması

T4, BF16 odaklı yeni nesil Tensor Core donanımı olmadığı için compute dtype olarak **FP16** kullanıyoruz.

`bnb_4bit_use_double_quant=True`, quantization sabitlerini de sıkıştırarak ek bellek tasarrufu sağlar.

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.8-27B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Model:", MODEL_ID)
print(bnb_config)

## 4. Modeli yükle

Bu hücre gerçek deneydir. T4'e sığarsa devam ederiz.

OOM olursa:
1. Runtime'ı yeniden başlat.
2. Başka büyük değişken/model bırakma.
3. İsteğe bağlı CPU-offload bölümünü dene.
4. Yine sığmazsa sonucu `OOM` olarak kaydet ve 3–3.5 bit GGUF deneyine geç.

> Model indirmesi ve quantization Colab VM üzerinde yapılır.

In [ ]:
def cuda_mem():
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_gb": torch.cuda.memory_allocated() / 1024**3,
        "reserved_gb": torch.cuda.memory_reserved() / 1024**3,
        "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

load_start = time.perf_counter()

processor = AutoProcessor.from_pretrained(MODEL_ID)

try:
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    LOAD_OK = True
    LOAD_ERROR = None
except Exception as e:
    LOAD_OK = False
    LOAD_ERROR = repr(e)
    print("MODEL YÜKLENEMEDİ:")
    print(LOAD_ERROR)
    raise

load_seconds = time.perf_counter() - load_start

print(f"Yükleme süresi: {load_seconds:.1f} sn")
print("Device map:", getattr(model, "hf_device_map", None))
print("CUDA memory:", {k: round(v, 3) for k, v in cuda_mem().items()})

## 5. Model bellek ayak izi

Transformers'ın raporladığı model footprint'i ile CUDA ölçümünü birlikte kaydediyoruz.

In [ ]:
footprint_gb = None
if hasattr(model, "get_memory_footprint"):
    footprint_gb = model.get_memory_footprint() / 1024**3

print("Model memory footprint:", None if footprint_gb is None else round(footprint_gb, 3), "GiB")
print("CUDA:", {k: round(v, 3) for k, v in cuda_mem().items()})

## 6. Text-only inference testi

Qwen3.8 multimodal bir model olsa da önce **text-only** test yapıyoruz.
Görüntü işleme ek belleği devreye sokmadan dil modeli tarafının T4 sınırını görmek daha temiz bir ilk deneydir.

In [ ]:
def build_inputs(prompt: str):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt}
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # İlk hesaplama cihazı CUDA olmalı. Device-map kullanılsa da girişler ilk GPU'ya taşınır.
    return inputs.to("cuda")


@torch.inference_mode()
def generate_once(prompt, max_new_tokens=64, do_sample=False):
    inputs = build_inputs(prompt)
    input_tokens = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        use_cache=True,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    new_tokens = outputs.shape[-1] - input_tokens
    tps = new_tokens / elapsed if elapsed > 0 else float("nan")
    peak = torch.cuda.max_memory_allocated() / 1024**3

    text = processor.decode(
        outputs[0][input_tokens:],
        skip_special_tokens=True
    )

    return {
        "prompt": prompt,
        "output": text,
        "input_tokens": int(input_tokens),
        "new_tokens": int(new_tokens),
        "elapsed_s": elapsed,
        "tokens_per_s": tps,
        "peak_vram_gb": peak,
    }


test = generate_once(
    "Model quantization nedir? INT4 quantization'ın avantajını iki kısa cümleyle açıkla.",
    max_new_tokens=64,
)

for k, v in test.items():
    if k != "output":
        print(k, "=", round(v, 3) if isinstance(v, float) else v)

print("\nÇIKTI:\n", test["output"])

## 7. Benchmark

Aynı ayarlarla birkaç kısa prompt çalıştırıp:

- toplam süre,
- üretilen token sayısı,
- tokens/s,
- peak VRAM

ölçüyoruz.

T4 çok sınırda olduğu için `max_new_tokens=64` ile başlıyoruz.

In [ ]:
PROMPTS = [
    "Python'da binary search algoritmasını kısa ve doğru şekilde açıkla.",
    "Quantization ile pruning arasındaki fark nedir? Kısa cevap ver.",
    "Bir yapay zeka modelinde VRAM kullanımını etkileyen üç ana faktörü say.",
]

results = []

# İlk çağrı CUDA kernel warm-up etkisi taşıyabilir.
_ = generate_once("Bir kelimeyle cevap ver: 2+2 kaçtır?", max_new_tokens=8)

for i, prompt in enumerate(PROMPTS, 1):
    print(f"\n--- Benchmark {i}/{len(PROMPTS)} ---")
    try:
        r = generate_once(prompt, max_new_tokens=64)
        results.append(r)
        print("tokens/s:", round(r["tokens_per_s"], 3))
        print("peak VRAM:", round(r["peak_vram_gb"], 3), "GB")
        print("çıktı:", r["output"][:300])
    except torch.OutOfMemoryError as e:
        print("OOM:", e)
        torch.cuda.empty_cache()
        results.append({
            "prompt": prompt,
            "status": "OOM",
            "error": str(e),
        })

print("\nTamamlandı.")

In [ ]:
valid = [r for r in results if "tokens_per_s" in r]

if valid:
    avg_tps = sum(r["tokens_per_s"] for r in valid) / len(valid)
    max_peak = max(r["peak_vram_gb"] for r in valid)

    print("Başarılı koşu:", len(valid), "/", len(results))
    print("Ortalama tokens/s:", round(avg_tps, 3))
    print("Maksimum peak VRAM:", round(max_peak, 3), "GB")
else:
    print("Başarılı inference yok; OOM sonucu proje verisi olarak kaydedilmeli.")

## 8. Sonuçları dosyaya kaydet

Colab oturumu kapanmadan sonuçları JSON olarak kaydedebilirsin.
İstersen daha sonra Drive'a manuel kopyalayabilirsin.

In [ ]:
summary = {
    "model": MODEL_ID,
    "gpu": torch.cuda.get_device_name(0),
    "total_vram_gb": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "quantization": "bitsandbytes 4-bit NF4 + double quant + FP16 compute",
    "load_seconds": load_seconds,
    "model_footprint_gib": footprint_gb,
    "results": results,
}

with open("qwen3_8_27b_t4_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Kaydedildi: qwen3_8_27b_t4_results.json")

## 9. İsteğe bağlı: CPU offload deneyi

**Sadece 4-bit doğrudan yükleme OOM verirse** ve Colab oturumunda yeterli sistem RAM'i varsa dene.

28B modelde CPU'ya taşınan katmanlar büyük miktarda RAM tüketebilir. Standart düşük-RAM Colab oturumunda
bu yöntem de başarısız olabilir. Bu da benchmark açısından geçerli bir sonuçtur.

Bu hücreyi çalıştırmadan önce Runtime'ı yeniden başlatıp yalnızca gerekli hücreleri çalıştırmak en temiz yöntemdir.

In [ ]:
# İSTEĞE BAĞLI / OOM SONRASI DENE
#
# from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
# import torch
#
# MODEL_ID = "Qwen/Qwen3.8-27B"
# processor = AutoProcessor.from_pretrained(MODEL_ID)
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
#
# model = AutoModelForMultimodalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     max_memory={0: "14GiB", "cpu": "45GiB"},
#     low_cpu_mem_usage=True,
# )

## 10. Proje yorumlama rehberi

Raporda sadece “çalıştı / çalışmadı” demek yerine şu sonucu çıkar:

| Deney | Bit | Peak VRAM | Load Time | Tokens/s | Sonuç |
|---|---:|---:|---:|---:|---|
| FP16 teorik | 16 | ~52 GiB sadece ağırlık | — | — | T4'e sığmaz |
| NF4 | 4 | ölç | ölç | ölç | başarılı / OOM |
| NF4 + offload | 4 | ölç | ölç | ölç | başarılı / OOM |
| 3–3.5 bit GGUF | ~3–3.5 | sonraki deney | ölç | ölç | sonraki aşama |

**Ana araştırma sorusu:**  
“Yaklaşık 28B parametreli Qwen3.8-27B modeli için 16 GB NVIDIA T4 üzerinde uygulanabilir en düşük maliyetli quantization/deployment ayarı nedir ve bunun hız-bellek-kalite dengesi nasıldır?”

## Kaynak notu

Model yükleme sınıfı ve chat-template kullanımı Qwen'in resmi Hugging Face model kartındaki Transformers örneğine,
4-bit NF4 ayarları ise Hugging Face `bitsandbytes` quantization dokümantasyonuna göre hazırlanmıştır.